[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-10-langchain-agents.ipynb#scrollTo=a1b2c3d4)

---
# Day 10 · LangChain Agents — ReAct Loop and Tool Use
**certified-journeys / llm-engineering-certified** · Day 10 · Agents & Tool Use

> **Goal for today:** Build a ReAct agent with multiple tools, trace each Thought/Action/Observation step, and handle agent errors gracefully.


In [ ]:
%pip install -q langchain langchain-openai langchain-community wikipedia duckduckgo-search


## Step 1 · The ReAct Loop — How Agents Think

The **ReAct** (Reasoning + Acting) loop is the core of LangChain agents:

| Phase | What happens |
|---|---|
| **Thought** | LLM reasons about what to do next |
| **Action** | LLM picks a tool and its input |
| **Observation** | Tool runs; result is fed back to LLM |
| **Repeat** | Until LLM outputs `Final Answer` |

The loop terminates when the model decides it has enough information to answer or when `max_iterations` is hit.


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool
from langchain import hub

# Set your API key — in Colab use: os.environ["OPENAI_API_KEY"] = "sk-..."
# For this demo we'll mock the LLM to avoid requiring a key
# In real usage: llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imports successful")
print("Required env var: OPENAI_API_KEY")


### What just happened?
- **`ChatOpenAI`** is the LLM backbone — it generates the Thought and Action text.
- **`create_react_agent`** wires the LLM to a ReAct prompt template.
- **`AgentExecutor`** is the runtime loop that calls tools and feeds observations back.
- **`DuckDuckGoSearchRun`** is a free built-in tool — no API key needed.


## Step 2 · Define Tools and Build the Agent

An agent needs at least two things:
1. **A list of tools** — each tool has a `name`, `description`, and callable `func`.
2. **A ReAct prompt** — tells the model the tool names and the Thought/Action/Observation format.

We pull the standard ReAct prompt from LangChain Hub (`hwchase17/react`) which is the canonical template.


In [ ]:
# --- Tool 1: Web search via DuckDuckGo (free, no key needed) ---
search_tool = DuckDuckGoSearchRun()

# --- Tool 2: Simple arithmetic evaluator ---
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the numeric result.
    Input must be a valid Python arithmetic expression, e.g. '(1024 * 3) / 2'.
    Do NOT pass word problems — extract the numbers first.
    """
    try:
        # eval is safe here because we restrict to math ops in a sandboxed env
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"

tools = [search_tool, calculator]

# List tool names and descriptions — the agent uses these to decide which tool to call
for t in tools:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description[:80]}...\n")


### What just happened?
- **`@tool`** decorator turns a plain Python function into a LangChain tool automatically.
- The **docstring** becomes the tool's `description` — the agent reads it to decide when to use the tool.
- **`tool.name`** defaults to the function name (`calculator`).
- We kept `eval` sandboxed by passing empty `__builtins__` — safe for math expressions.


## Step 3 · Wire the Agent with create_react_agent

`create_react_agent` is the recommended factory in LangChain v0.1+. It replaces the older `initialize_agent`.

The flow: **LLM + tools + ReAct prompt template → agent** → wrapped in `AgentExecutor` to actually run.


In [ ]:
from langchain import hub

# Pull the canonical ReAct prompt from LangChain Hub
# This template formats the Thought/Action/Observation trace correctly
react_prompt = hub.pull("hwchase17/react")

# Show the prompt template variables — agent needs 'tools', 'tool_names', 'input', 'agent_scratchpad'
print("Prompt input variables:", react_prompt.input_variables)
print("\nFirst 300 chars of template:")
print(str(react_prompt.messages[0].prompt.template)[:300] if hasattr(react_prompt, 'messages') else str(react_prompt)[:300])


In [ ]:
# Build the agent — requires OPENAI_API_KEY to be set
# Uncomment and run when you have a key:

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)
# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=tools,
#     verbose=True,          # prints each Thought/Action/Observation step
#     max_iterations=6,      # safety cap — prevents infinite loops
#     handle_parsing_errors=True,  # gracefully handle malformed LLM output
# )
# print("Agent built:", type(agent_executor))
# print("Registered tools:", [t.name for t in agent_executor.tools])

# --- Demo output (without API key) ---
print("AgentExecutor config (demo):")
print("  verbose=True       → prints Thought/Action/Observation on each loop")
print("  max_iterations=6   → stops after 6 ReAct cycles")
print("  handle_parsing_errors=True → LLM format errors become Observations")
print("  tools registered:", [t.name for t in tools])


### What just happened?
- **`create_react_agent`** returns a `RunnableAgent` — a chain, not an executor.
- **`AgentExecutor`** wraps it with the tool-calling loop and safety params.
- **`verbose=True`** is the key development flag — you see every step the agent takes.
- **`handle_parsing_errors=True`** catches cases where the LLM doesn't follow the Action format.


## Step 4 · Run the Agent on a Multi-Step Question

A good test question for a 2-tool agent requires **both** tools in sequence:
1. Search for a fact (a number)
2. Apply arithmetic to that number

With `verbose=True` you'll see each Thought/Action/Observation printed to stdout.


In [ ]:
# Multi-step question: requires search (to find population) then arithmetic
question = "What is the population of Tokyo? If each person consumed 2.5 kg of rice per month, how many kg would the city consume in a year?"

# With a live API key, run:
# result = agent_executor.invoke({"input": question})
# print("\nFinal answer:", result["output"])

# --- Simulated trace output (matches real verbose=True output structure) ---
simulated_trace = """
> Entering new AgentExecutor chain...

Thought: I need to find Tokyo's population first, then calculate rice consumption.
Action: duckduckgo_search
Action Input: Tokyo population 2024
Observation: Tokyo's population is approximately 13.96 million (city) or 37.4 million (metro area) as of 2024.

Thought: I'll use the city population of 13,960,000. Now I calculate:
         13,960,000 people × 2.5 kg/month × 12 months = ? kg/year
Action: calculator
Action Input: 13960000 * 2.5 * 12
Observation: 418800000.0

Thought: I now have the final answer.
Final Answer: Tokyo's population is approximately 13.96 million people.
At 2.5 kg of rice per person per month, the city would consume
418,800,000 kg (418,800 metric tons) of rice per year.

> Finished chain.
"""
print(simulated_trace)


### What just happened?
- The agent issued **two tool calls** in sequence — search first, then calculator.
- Each **Observation** was fed back into the LLM context, building a running scratchpad.
- The agent extracted the number from prose ("13.96 million") and converted it to `13960000` for the calculator.
- **`Final Answer:`** signals the loop to stop and return the output string.


## Step 5 · Tracing Thought/Action/Observation Steps

Beyond `verbose=True`, LangChain supports structured **callbacks** that let you capture each step programmatically — useful for logging, debugging, or building custom UIs.


In [ ]:
from langchain.callbacks.base import BaseCallbackHandler
from typing import Any, Dict, List, Union

class StepTracer(BaseCallbackHandler):
    """Captures every agent step into a structured list for inspection."""

    def __init__(self):
        self.steps = []
        self.step_num = 0

    def on_agent_action(self, action, **kwargs):
        """Called when the agent decides to use a tool."""
        self.step_num += 1
        self.steps.append({
            "step": self.step_num,
            "type": "action",
            "tool": action.tool,
            "input": action.tool_input,
            "log": action.log.strip()[:200],  # Thought text
        })
        print(f"[Step {self.step_num}] Action → {action.tool}({action.tool_input!r})")

    def on_tool_end(self, output: str, **kwargs):
        """Called when a tool returns its result."""
        self.steps.append({"step": self.step_num, "type": "observation", "output": output[:200]})
        print(f"[Step {self.step_num}] Observation → {output[:100]}")

    def on_agent_finish(self, finish, **kwargs):
        """Called when the agent outputs Final Answer."""
        self.steps.append({"step": "final", "type": "finish", "output": finish.return_values})
        print(f"[Final] {finish.return_values}")

# Usage with a live executor:
# tracer = StepTracer()
# result = agent_executor.invoke({"input": question}, config={"callbacks": [tracer]})
# print(f"\nTotal steps taken: {tracer.step_num}")
# for s in tracer.steps:
#     print(s)

# Demo — show what the tracer captures
tracer_demo = StepTracer()
tracer_demo.step_num = 0

from langchain.schema.agent import AgentAction, AgentFinish
tracer_demo.on_agent_action(AgentAction(tool="duckduckgo_search", tool_input="Tokyo population 2024", log="Thought: I need the population first."))
tracer_demo.on_tool_end("Tokyo population is approximately 13.96 million.")
tracer_demo.on_agent_action(AgentAction(tool="calculator", tool_input="13960000 * 2.5 * 12", log="Thought: Now I calculate rice consumption."))
tracer_demo.on_tool_end("418800000.0")
tracer_demo.on_agent_finish(AgentFinish(return_values={"output": "418,800,000 kg per year"}, log=""))

print(f"\nCaptured {len(tracer_demo.steps)} steps")


### What just happened?
- **`BaseCallbackHandler`** is a hook system — override methods to intercept any LangChain event.
- **`on_agent_action`** fires before each tool call; **`on_tool_end`** fires after.
- Storing steps in a list lets you post-process the trace (count steps, detect loops, log to a DB).
- **This is how LangSmith works** internally — same callback hooks, just sent to a remote server.


## Step 6 · Error Handling — max_iterations and Timeouts

Agents can get stuck in loops or exhaust your token budget. Two safeguards:

| Safeguard | Parameter | What it does |
|---|---|---|
| Iteration cap | `max_iterations=N` | Raises `OutputParserException` after N cycles |
| Time cap | `max_execution_time=seconds` | Hard wall-clock limit |
| Format errors | `handle_parsing_errors=True` | Malformed output becomes an Observation, not a crash |


In [ ]:
from langchain.agents import AgentExecutor
from langchain.schema import OutputParserException

def run_agent_safely(executor: AgentExecutor, question: str) -> dict:
    """Run an agent and handle the two most common failure modes."""
    try:
        result = executor.invoke({"input": question})
        return {"status": "ok", "output": result["output"]}

    except OutputParserException as e:
        # Triggered when max_iterations is hit and handle_parsing_errors=False
        return {
            "status": "max_iterations_hit",
            "output": "Agent could not complete in the allowed steps.",
            "error": str(e)[:200],
        }

    except Exception as e:
        # Catch-all for network errors, API timeouts, etc.
        return {
            "status": "error",
            "output": "Agent encountered an unexpected error.",
            "error": str(e)[:200],
        }

# Show the config that prevents runaway agents
safe_config = {
    "max_iterations": 6,           # stop after 6 ReAct cycles
    "max_execution_time": 30,       # stop after 30 seconds wall time
    "handle_parsing_errors": True,  # malformed LLM output → Observation, not crash
    "early_stopping_method": "generate",  # when max hit, ask LLM for best answer so far
}

print("Safe AgentExecutor configuration:")
for k, v in safe_config.items():
    print(f"  {k}: {v}")

# With a live executor:
# safe_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, **safe_config)
# result = run_agent_safely(safe_executor, "What is the meaning of life?")
# print(result)


### What just happened?
- **`max_iterations=6`** is a good default for most agents — complex tasks rarely need more than 5 cycles.
- **`early_stopping_method="generate"`** asks the LLM for its best partial answer instead of just raising an error.
- **`handle_parsing_errors=True`** is especially important when using smaller, less instruction-following models.
- Always wrap `executor.invoke` in a try/except in production — agents can fail for many external reasons.


## Step 7 · Inspecting Registered Tools

After building an executor, you can introspect its tool registry programmatically — useful for debugging and validation.


In [ ]:
def inspect_tools(tools_list: list) -> None:
    """Print a formatted summary of all registered tools."""
    print(f"{'='*60}")
    print(f"Registered tools: {len(tools_list)}")
    print(f"{'='*60}")
    for i, t in enumerate(tools_list, 1):
        print(f"\n[{i}] {t.name}")
        print(f"    Description : {t.description[:120]}")
        # Show argument schema if available (StructuredTool)
        if hasattr(t, 'args_schema') and t.args_schema:
            print(f"    Args schema : {t.args_schema.schema()}")
        else:
            print(f"    Args        : single string input")
        print(f"    Return type : str (all LangChain tools return strings)")

inspect_tools(tools)

# From a live AgentExecutor: inspect_tools(agent_executor.tools)


### What just happened?
- **`agent_executor.tools`** is a plain Python list — iterate it like any list.
- **`t.description`** is the exact string the LLM sees in its system prompt when deciding which tool to use.
- **`args_schema`** is populated for `StructuredTool` (Day 11); plain `@tool` functions use a single string.
- All LangChain tools **must return a string** — the executor converts the return value to str automatically.


## Step 8 · Adding a Custom Memory-Style Tool

Beyond search and calculation, agents often need a "notepad" tool — a way to store intermediate results across steps so they don't re-query the same data.


In [ ]:
from langchain.tools import tool

# A simple in-process notepad the agent can write to and read from
_notepad: dict[str, str] = {}

@tool
def notepad_write(entry: str) -> str:
    """Save a key-value pair to the notepad for later retrieval.
    Input format: 'key::value' (use double colon as separator).
    Example: 'tokyo_population::13960000'
    Returns a confirmation message.
    """
    if "::" not in entry:
        return "Error: use 'key::value' format."
    key, value = entry.split("::", 1)
    _notepad[key.strip()] = value.strip()
    return f"Saved: {key.strip()} = {value.strip()}"

@tool
def notepad_read(key: str) -> str:
    """Read a previously saved value from the notepad by key.
    Returns the value, or 'Not found' if the key doesn't exist.
    Use this to retrieve intermediate results saved in earlier steps.
    """
    return _notepad.get(key.strip(), f"Not found: '{key.strip()}'")

# Test the notepad tools directly
print(notepad_write.invoke("tokyo_population::13960000"))
print(notepad_read.invoke("tokyo_population"))
print(notepad_read.invoke("nonexistent_key"))

# Extended tool list with notepad
extended_tools = [search_tool, calculator, notepad_write, notepad_read]
print(f"\nExtended tool registry: {[t.name for t in extended_tools]}")


### What just happened?
- **Notepad tools** let the agent store facts across steps without re-querying — saves tokens and reduces errors.
- The `::` separator convention is documented in the docstring — the agent learns the format from it.
- **State is shared** between `notepad_write` and `notepad_read` via the module-level `_notepad` dict.
- In production, replace the in-memory dict with Redis or a DB for persistence across sessions.


In [ ]:
# Challenge: Build a two-tool ReAct agent that answers a chained question
# Your task:
#   1. Create a @tool function called `unit_converter` that converts between
#      metric and imperial units. Accept input like "100 kg to lbs".
#   2. Create a second @tool that counts the number of words in a string.
#   3. Build an AgentExecutor with both tools + DuckDuckGoSearchRun.
#   4. Run the agent on: "How many words are in the Wikipedia article title
#      for the heaviest land animal? Also convert its weight from kg to lbs."
# Hints:
#   - unit_converter: 1 kg ≈ 2.20462 lbs; use float parsing with regex
#   - word_counter: split on whitespace
#   - Set max_iterations=8 and verbose=True

# Your solution here
@tool
def unit_converter(query: str) -> str:
    """Convert between common units. Input: '<number> <from_unit> to <to_unit>'.
    Supported: kg↔lbs, km↔miles, celsius↔fahrenheit.
    """
    # TODO: implement conversion logic using regex to parse query
    pass

@tool
def word_counter(text: str) -> str:
    """Count the number of words in the provided text string.
    Returns the count as a string, e.g. '5 words'.
    """
    # TODO: implement word counting
    pass


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| ReAct loop | Thought → Action → Observation, repeat until Final Answer |
| `create_react_agent` | Factory that wires LLM + tools + prompt; returns a Runnable |
| `AgentExecutor` | Runtime loop; set `verbose=True`, `max_iterations`, `handle_parsing_errors` |
| `@tool` decorator | Turns any Python function into a LangChain tool; docstring = description |
| `BaseCallbackHandler` | Hook into every agent event (action, observation, finish) |
| Tool description quality | Agent picks tools based solely on descriptions — be specific |

> **Tip:** Set verbose=True while developing agents — blind agents fail silently. Only turn it off in production.

---
## What's next
**Day 11** → Custom Tools with `@tool`, `StructuredTool`, Pydantic schemas, and DuckDB integration — plus testing tool failure recovery.

Mark Day 10 complete in your [tracker](../index.html).
